In [1]:
import wget
from zipp import zipfile
from pathlib import Path
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import pandas as pd
import csv
import json
import os

import warnings
warnings.simplefilter("ignore") # centroid gives : UserWarning: Geometry is in a geographic CRS

import requests
from bs4 import BeautifulSoup
import numpy as np

import folium

In [2]:
REGION_STRUCTURE_FILE = Path("../climatechangeimpact/region_structure.json")
DONE_ROOT = Path("../climatechangeimpact/done")

def load_region_structure(path: Path = REGION_STRUCTURE_FILE) -> dict[str, list[str]]:
    """Load the {country: [camel_id, ...]} mapping of all possible regions."""
    with open(path, "r") as f:
        return json.load(f)


def load_done_regions(country: str, done_root: Path = DONE_ROOT) -> set[str]:
    """Read finished camel_ids from done/<country>/regions_done.csv."""
    csv_path = done_root / country / "regions_done.csv"
    done = set()
    if not csv_path.exists():
        return done

    with open(csv_path, "r", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)  # skip header
        for row in reader:
            if row:
                done.add(row[0])
    return done


def get_todo_regions(country: str, structure: dict[str, list[str]]) -> list[str]:
    """Return camel_ids for a country that are NOT yet marked done."""
    all_ids = set(structure.get(country, []))
    done_ids = load_done_regions(country)
    return sorted(all_ids - done_ids)

structure = load_region_structure()
total_all = total_done = total_todo = 0

for c, ids in structure.items():
    done = load_done_regions(c)
    todo = set(ids) - done
    print(f"{c:<28} {len(ids):>6} {len(done):>6} {len(todo):>6}")
    total_all  += len(ids)
    total_done += len(done)
    total_todo += len(todo)

print(f"{'TOTAL':<28} {total_all:>6} {total_done:>6} {total_todo:>6}")

australia                       150     74     76
austria                         307    143    164
brazil                          376     40    336
canada                          886     42    844
chile                           314     35    279
czech_republic                   35     17     18
england                         247     32    215
germany                         120     28     92
lichtenstein                      1      1      0
mexico                           16      6     10
scotland                        124     22    102
switzerland                      16     15      1
united_states_of_america       4201    138   4063
wales                            37     21     16
TOTAL                          6830    614   6216


In [3]:
LINK = "https://github.com/eWaterCycle/CCI-analysis-seamless/blob/main/climatechangeimpact/regions/"

In [4]:
TOC_FILE = Path("_toc.yml")
LINK_ROOT = "https://github.com/eWaterCycle/CCI-analysis-seamless/blob/main/climatechangeimpact"

# Read existing _toc.yml as raw text
with open(TOC_FILE, "r") as f:
    toc_text = f.read()

# Find the "- caption: Regions" block and cut it (and everything after) off
marker = "  - caption: Regions"
idx = toc_text.find(marker)
if idx == -1:
    raise ValueError("Could not find '- caption: Regions' in _toc.yml")
header = toc_text[:idx].rstrip() + "\n"

# Build the new Regions section from done CSVs
lines = ["  - caption: Regions", "    chapters:"]
for country in sorted(structure.keys()):
    done_ids = sorted(load_done_regions(country))
    if not done_ids:
        continue
    lines.append(f"      - external: {LINK_ROOT}/regions/{country}.md")
    lines.append(f"        sections:")
    for camel_id in done_ids:
        lines.append(
            f"          - external: {LINK_ROOT}/regions/{country}/{camel_id}/step_4_analyse_executed.ipynb"
        )

new_toc = header + "\n".join(lines) + "\n"

with open(TOC_FILE, "w") as f:
    f.write(new_toc)

print(f"Wrote {TOC_FILE} ({sum(len(load_done_regions(c)) for c in structure)} done regions across {sum(1 for c in structure if load_done_regions(c))} countries)")


Wrote _toc.yml (614 done regions across 14 countries)
